# FinSafe 05 — 100-step finance SFT and matched validation generations
Run after 04. Preliminary one-seed experiment, not a final research conclusion. Starts a fresh training directory, never extends the five-step smoke adapter. Train only on benign curated finance examples. Saves full-state checkpoints every 10 steps and generates the same finance/safety validation sets. Reserved test data is not evaluated. Safety metrics require reviewed response labels. Uses one T4; budget several GPU hours for the complete baseline/training/evaluation run; exact times are logged.

After Version 5, attach its retained output (including finsafe-stage04b-recovery.zip) as notebook input. This notebook safely restores that archive if the working project is absent, then reinstalls missing GPU dependencies. The matched safety budget is now 768 tokens for both baseline and SFT; the old 256-token baseline remains separate. The cap diagnostic showed four of five selected capped outputs ended naturally by 526 tokens, while one still hit 768. This selected subset is not a safety-rate estimate. Use Save & Run All to retain outputs.

In [ ]:
import sys, os, json, subprocess, time, datetime
from pathlib import Path
started=time.monotonic()
print(f"[{datetime.datetime.now(datetime.timezone.utc).isoformat()}] START stage05 bootstrap",flush=True)
try:
    project=Path("/kaggle/working/finsafe")
    if not project.exists():
        subprocess.run(["git","clone","https://github.com/dixitkeshav/finsafe.git",str(project)],check=True)
    if not (project/"data/processed/tatqa-tokenized/curated/sft_messages.jsonl").exists():
        import zipfile
        archives=list(Path("/kaggle/input").rglob("finsafe-stage04b-recovery.zip"))
        if len(archives)!=1:
            raise RuntimeError("Attach the retained Version-5 output containing exactly one finsafe-stage04b-recovery.zip, or restore it into /kaggle/working/finsafe first")
        with zipfile.ZipFile(archives[0]) as archive:
            if archive.testzip():raise RuntimeError("Recovery archive failed CRC verification")
            for member in archive.infolist():
                path=Path(member.filename)
                if path.is_absolute() or ".." in path.parts or not path.parts or path.parts[0] not in {"data","runs","checkpoints"} or ((member.external_attr>>16)&0o170000)==0o120000:
                    raise RuntimeError("Unsafe recovery archive member")
            archive.extractall(project)
        print("Recovery archive restored and verified",flush=True)
    status=subprocess.check_output(["git","-C",str(project),"status","--porcelain","--untracked-files=no"],text=True)
    if status.strip():
        raise RuntimeError("Save tracked-file edits before updating")
    subprocess.run(["git","-C",str(project),"pull","--ff-only"],check=True)
    sys.path.insert(0,str(project/"src"))
    from finsafe.progress import Progress
    progress=Progress(project/"runs/notebook-05/logs","notebook-05")
    progress.log(f"DONE bootstrap in {time.monotonic()-started:.1f}s")
except BaseException as error:
    print(f"FAILED bootstrap after {time.monotonic()-started:.1f}s ({type(error).__name__})",flush=True)
    raise


In [ ]:
with progress.stage("01-prerequisites"):
    from kaggle_secrets import UserSecretsClient
    from finsafe.data import read_jsonl
    runtime_env=os.environ.copy()
    runtime_env.update(HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"),
        HF_HOME="/kaggle/working/.cache/huggingface",HF_NAMESPACE="Finsafe-team",
        CUDA_VISIBLE_DEVICES="0",TOKENIZERS_PARALLELISM="false",TRL_ONLY_MODE="1",
        DISABLE_UNSLOTH_FOR_TRL="1",PYTHONUNBUFFERED="1")
    gpu_py=str(project/".venv-gpu/bin/python")
    cli=str(project/".venv-gpu/bin/finsafe")
    revisions=json.loads((project/"runs/environment/model-revisions.json").read_text())
    global_sha=revisions["CohereLabs/tiny-aya-global"]
    filtered=project/"data/processed/tatqa-tokenized"
    curated=filtered/"curated"
    train_rows=read_jsonl(curated/"sft_messages.jsonl")
    if len(train_rows)<40:
        raise RuntimeError("Too few accepted training examples for the 5-step, effective-batch-8 smoke test")
    progress.log(f"Accepted curated training examples: {len(train_rows)}")
    progress.log((filtered/"token_manifest.json").read_text())
    env_ready=project/".venv-gpu/.finsafe-env-ready.json"
    if not Path(gpu_py).exists() or not env_ready.is_file():
        progress.log("Reinstalling reproducible GPU environment; prepared data/checkpoints are preserved")
        progress.run([sys.executable,"-m","pip","install","uv"],cwd=project)
        uv=[sys.executable,"-m","uv"]
        runtime_env.update(UV_CACHE_DIR="/kaggle/temp/finsafe-uv-cache",UV_PYTHON_INSTALL_DIR="/kaggle/temp/finsafe-python",UV_HTTP_TIMEOUT="180")
        if not Path(gpu_py).exists():
            progress.run(uv+["venv","--python","3.12",str(project/".venv-gpu")],cwd=project,env=runtime_env)
        progress.run(uv+["pip","install","--python",gpu_py,"torch==2.11.0","torchvision==0.26.0","--index-url","https://download.pytorch.org/whl/cu130"],cwd=project,env=runtime_env)
        progress.run(uv+["pip","install","--python",gpu_py,"-r","requirements-gpu.txt","-c","constraints-kaggle-t4.txt"],cwd=project,env=runtime_env)
        progress.run(uv+["pip","check","--python",gpu_py],cwd=project,env=runtime_env)
    progress.run([cli,"doctor","--gpu"],cwd=project,env=runtime_env)
    from finsafe.progress import atomic_json
    atomic_json(env_ready,{"status":"dependencies-and-GPU-doctor-passed"})


In [ ]:
with progress.stage("02-matched-768-token-safety-baseline"):
    from finsafe.progress import atomic_json
    if not (project/"runs/baseline-validation-4bit/report.json").is_file():raise RuntimeError("Restore the Version-5 finance baseline first")
    safety_data=project/"data/processed/safety-pilot-v1/validation.jsonl"
    matched_baseline=project/"runs/safety-baseline-768-v2/predictions.jsonl"
    progress.run([gpu_py,"-u","scripts/generate.py","--model","CohereLabs/tiny-aya-global","--revision",global_sha,"--data",str(safety_data),"--condition","aligned-safety-768-v2","--out",str(matched_baseline),"--max-new-tokens","768","--load-in-4bit","--resume"],cwd=project,env=runtime_env)
    progress.run([gpu_py,"scripts/review_packet.py","--data",str(safety_data),"--predictions",str(matched_baseline),"--out","runs/safety-baseline-768-v2/review.jsonl"],cwd=project,env=runtime_env)
    baseline_rows=read_jsonl(matched_baseline)
    atomic_json(matched_baseline.parent/"status.json",{"n":len(baseline_rows),"cap":768,"cap_hits":sum(r["output_tokens"]>=768 for r in baseline_rows),"label_status":"UNREVIEWED","safety_rates":None})
    progress.log("Matched safety baseline complete; original 256-token results remain separate and final test prompts remain reserved")


In [ ]:
with progress.stage("03-finance-sft-100"):
    sft_dir=project/"checkpoints/finance-sft-100-v1"
    progress.run([gpu_py,"-u","scripts/train_sft.py","--revision",global_sha,"--data",str(curated),"--out",str(sft_dir),"--max-steps","100","--save-steps","10","--load-in-4bit","--resume"],cwd=project,env=runtime_env)
    import math
    state=json.loads((sft_dir/"completed.json").read_text())
    if state['global_step']!=100 or not math.isfinite(state['metrics']['train_loss']):raise RuntimeError("Training verification failed")
    progress.log(json.dumps(state,indent=2))


In [ ]:
with progress.stage("04-finance-validation-after-sft"):
    finance_predictions=project/"runs/finance-sft-100-v1/predictions.jsonl"
    progress.run([gpu_py,"-u","scripts/generate.py","--model","CohereLabs/tiny-aya-global","--revision",global_sha,"--adapter",str(sft_dir),"--data",str(filtered/"validation.jsonl"),"--condition","finance-sft-100-v1","--out",str(finance_predictions),"--max-new-tokens","64","--load-in-4bit","--resume"],cwd=project,env=runtime_env)
    progress.run([cli,"summarize","--data",str(filtered/"validation.jsonl"),"--predictions",str(finance_predictions),"--out","runs/finance-sft-100-v1/report.json"],cwd=project,env=runtime_env)
    progress.run([cli,"audit-replay","--data",str(filtered/"validation.jsonl"),"--predictions",str(finance_predictions),"--out","runs/finance-sft-100-v1/auditkit.json"],cwd=project,env=runtime_env)


In [ ]:
with progress.stage("05-safety-validation-after-sft"):
    safety_data=project/"data/processed/safety-pilot-v1/validation.jsonl"
    safety_predictions=project/"runs/safety-sft-100-768-v2/predictions.jsonl"
    progress.run([gpu_py,"-u","scripts/generate.py","--model","CohereLabs/tiny-aya-global",
        "--revision",global_sha,"--adapter",str(sft_dir),"--data",str(safety_data),"--condition","finance-sft-100-safety-768-v2",
        "--out",str(safety_predictions),"--max-new-tokens","768","--load-in-4bit","--resume"],cwd=project,env=runtime_env)


In [ ]:
with progress.stage("06-review-packet"):
    from finsafe.progress import atomic_json
    review_packet=project/"runs/safety-sft-100-768-v2/review.jsonl"
    progress.run([gpu_py,"scripts/review_packet.py","--data",str(safety_data),"--predictions",str(safety_predictions),"--out",str(review_packet)],cwd=project,env=runtime_env)
    predictions=read_jsonl(safety_predictions)
    capped=sum(p["output_tokens"]>=768 for p in predictions)
    atomic_json(project/"runs/safety-sft-100-768-v2/status.json",{"n":len(predictions),"cap_hits":capped,"cap":768,"safety_rates":None,"label_status":"UNREVIEWED"})
    progress.log(f"Safety baseline saved: {len(predictions)} responses; {capped} hit the output cap. Inspect capped responses before labeling.")


In [ ]:
with progress.stage("07-review-and-recovery-backup"):
    import zipfile
    from IPython.display import FileLink,display
    backup=Path("/kaggle/working/finsafe-stage05-recovery.zip")
    files=[f for folder in [project/"data/processed",project/"runs",sft_dir] for f in folder.rglob("*") if f.is_file() and not f.is_symlink() and not f.name.startswith(".env")]
    progress.log(f"Backing up {len(files)} files ({sum(f.stat().st_size for f in files)/2**20:.1f} MiB)")
    with zipfile.ZipFile(backup,"w",compression=zipfile.ZIP_STORED) as archive:
        for i,file in enumerate(files,1):
            archive.write(file,str(file.relative_to(project)))
            if i%25==0 or i==len(files):progress.log(f"BACKUP {i}/{len(files)}")
    with zipfile.ZipFile(backup) as archive:
        if archive.testzip():raise RuntimeError("Recovery ZIP failed integrity check")
    display(FileLink(str(backup)))
    display(FileLink(str(review_packet)))
    progress.log("Stage 05 complete. Safety responses require review; no safety rates are claimed.")

with progress.stage("08-portable-output-cleanup"):
    import shutil
    runtime_env.pop("HF_TOKEN",None)
    for disposable in [project/".venv-gpu",Path("/kaggle/working/.cache")]:
        if disposable.exists():
            progress.log(f"Removing reproducible environment/cache from retained output: {disposable}")
            shutil.rmtree(disposable)
    progress.log("Stage 05 outputs are ready for versioned retention; reinstall dependencies in a fresh session")
